# Select and Configure Compute — TTC Service Complaints (Toronto Open Data theme)

This notebook is my own version of **Lab 02: Select and Configure Compute in Azure Databricks** from the Microsoft Learn *DP-750T00* course. The official lab uses a fictional healthcare scenario ("HealthBridge Analytics") generating synthetic patient admission records with the `faker` library.

Instead, I'm keeping the Toronto Transit theme from Lab 01: this notebook generates **synthetic TTC service complaint records** — fake rider names filing fake complaints against real TTC routes — to practice the same compute/library skills without using any real personal data.

**What's in this notebook (Exercises 3 & 4 — the notebook-based part of the lab):**
- Installing `faker` notebook-scoped with `%pip`
- Verifying the install with a Canadian-locale name + address
- Generating 100 synthetic TTC complaint records as a Spark DataFrame
- Analyzing complaint volume by route and by issue type

**Exercises 1 & 2 (cluster creation + cluster-scoped library install) happen in the Databricks UI, not in this notebook** — see the cluster settings I used, further down as a comment block, for reference.

> Attach this notebook to **Serverless** compute before running, per the lab instructions.

## Reference: cluster settings used for Exercises 1 & 2 (done in the Databricks UI)

| Setting | Value |
|---|---|
| Cluster name | `toronto-transit-dev` |
| Policy | Unrestricted |
| Cluster mode | Multi node |
| Access mode | Standard (formerly: Shared) |
| Databricks Runtime | Latest LTS |
| Worker type | Memory-optimized (e.g. E-series) |
| Autoscaling | Min 1, Max 3 workers |
| Auto termination | 15 minutes |
| Photon Acceleration | Enabled |

**Cluster-scoped library installed:** PyPI package `faker==40.8.0` (pinned version, same reproducibility reasoning as the original lab — consistent synthetic data generation across every run and every team member).

After confirming the library installed and the cluster reached **Running**, I stopped/deleted `toronto-transit-dev` since the remaining exercises run on Serverless compute.

## Exercise 3: Install `faker` Notebook-Scoped

### Task 3.1 — Install `faker` notebook-scoped

Installing notebook-scoped means the package applies only to this notebook session, without affecting other users or notebooks sharing the same compute.

In [ ]:
%pip install faker==40.8.0

### Task 3.2 — Verify the installation

Using the `en_CA` locale for a Toronto-appropriate touch: a random Canadian name and address.

In [ ]:
from faker import Faker

fake = Faker("en_CA")

print(f"Name: {fake.name()}")
print(f"Address: {fake.address()}")

## Exercise 4: Generate and Analyze Synthetic TTC Complaint Data

The TTC customer service team needs a synthetic dataset of rider complaints to test a new data pipeline. The data must look realistic enough to validate transformations and aggregations, but must contain **no real rider information** — same privacy principle as the original lab's HIPAA framing, applied to transit data instead of health data.

### Task 4.1 — Generate a synthetic complaints DataFrame

100 synthetic records, each with:

| Column | Description |
|---|---|
| `complaint_id` | Integer, 1 to 100 |
| `reporter_name` | Randomly generated full name (fake rider) |
| `filed_date` | Random date between `2023-01-01` and `2025-12-31`, as a string |
| `route` | One of 5 real TTC routes (randomly chosen) |
| `issue_type` | One of 5 issue types (randomly chosen) |

In [ ]:
import random

routes = ["504 King", "501 Queen", "510 Spadina", "29 Dufferin", "Line 1 Yonge-University"]
issue_types = ["Delay", "Overcrowding", "Mechanical Fault", "Missed Stop", "Rude Operator"]

records = []
for complaint_id in range(1, 101):
    records.append({
        "complaint_id": complaint_id,
        "reporter_name": fake.name(),
        "filed_date": str(fake.date_between(start_date="2023-01-01", end_date="2025-12-31")),
        "route": random.choice(routes),
        "issue_type": random.choice(issue_types),
    })

complaints_df = spark.createDataFrame(records)
display(complaints_df.limit(10))

### Task 4.2 — Analyze complaints by route

Which TTC routes generate the most complaints? Count complaints per `route`, ordered from most to least.

In [ ]:
from pyspark.sql.functions import col

route_counts = (
    complaints_df
    .groupBy("route")
    .count()
    .orderBy(col("count").desc())
)
display(route_counts)

## Bonus: Break down complaints by issue type

Same pattern, applied to `issue_type` instead of `route` — useful for spotting whether it's mostly delays, overcrowding, or something else.

In [ ]:
issue_counts = (
    complaints_df
    .groupBy("issue_type")
    .count()
    .orderBy(col("count").desc())
)
display(issue_counts)